In [1]:
import os
import pandas as pd
import numpy as np
import pickle

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"]="6"

# Load data

In [3]:
# File paths
DATA_DIR = "../data/ml-1m"
RATINGS_FILE = os.path.join(DATA_DIR, "ratings.dat")
USERS_FILE = os.path.join(DATA_DIR, "users.dat")
MOVIES_FILE = os.path.join(DATA_DIR, "movies.dat")

# Column names
ratings_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
users_cols = ['user_id', 'gender', 'age', 'occupation', 'zip_code']
movies_cols = ['movie_id', 'title', 'genres']

# Load data
ratings = pd.read_csv(RATINGS_FILE, sep='::', engine='python', names=ratings_cols, encoding='latin-1')
users = pd.read_csv(USERS_FILE, sep='::', engine='python', names=users_cols, encoding='latin-1')
movies = pd.read_csv(MOVIES_FILE, sep='::', engine='python', names=movies_cols, encoding='latin-1')

In [4]:
print(f"Ratings: {ratings.shape}")
print(f"Users: {users.shape}")
print(f"Movies: {movies.shape}")

Ratings: (1000209, 4)
Users: (6040, 5)
Movies: (3883, 3)


# Create dict of UserIDs-MovieRatingHistory

Keys: user_id (original ID) 

Values: List of (movie_id, rating) tuples in chronological order (all ratings 1-5)

In [5]:
sorted_ratings = ratings.sort_values(by=['user_id', 'timestamp'])

user_sequences = (
    sorted_ratings.groupby("user_id")
        .apply(lambda x: list(zip(x["movie_id"], x["rating"])))
        .to_dict()
)

In [6]:
print(f"Sorted ratings: {sorted_ratings.shape}")
print(f"# of users: {len(user_sequences)}")

# positive_ratings is a DataFrame. 
# 1000209 rows: 1 movie rating
# 4 columns: user_id, movie_id, rating, timestamp

# user_sequences is a dictionary
# Keys: UserIDs (ints)
# Values: list of tuples, each tuple is (movieID, rating)

Sorted ratings: (1000209, 4)
# of users: 6040


In [7]:
# save data to file
with open('../data/user_sequences.pkl', 'wb') as f:
    pickle.dump(user_sequences, f)

# Split 80/20

In [8]:
# Create one list of dataframes for training and one list of dataframes for testing
train_frames = []
test_frames = []

for user_id, all_user_movie_ratings in sorted_ratings.groupby('user_id'):
    # standard practice to use int, NOT round
    split_idx = int(len(all_user_movie_ratings) * 0.8)
    
    # Ensure at least one item in each if the user has enough ratings
    if len(all_user_movie_ratings) >= 2 and split_idx == 0:
        split_idx = 1

    per_user_train_split = all_user_movie_ratings.iloc[:split_idx]
    per_user_test_split = all_user_movie_ratings.iloc[split_idx:]
    
    train_frames.append(per_user_train_split)
    test_frames.append(per_user_test_split)

In [9]:
print(len(train_frames))
print(len(test_frames))
# Both are the # of users

6040
6040


In [10]:
# Turn each list of dataframes into 1 BIG dataframe
train_df = pd.concat(train_frames)
test_df = pd.concat(test_frames)

print(f"Train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")

Train size: 797758
Test size: 202451


In [11]:
# Create the dictionaries from the DataFrames we just made
train_sequences = train_df.groupby('user_id').apply(
    lambda x: list(zip(x['movie_id'], x['rating']))
).to_dict()

test_sequences = test_df.groupby('user_id').apply(
    lambda x: list(zip(x['movie_id'], x['rating']))
).to_dict()

# train_sequences and test_sequences are dictionaries like user_sequences
# Keys: UserIDs (ints)
# Values: list of tuples, each tuple is (movieID, rating)

In [12]:
# Save the split data
with open('../data/train_sequences.pkl', 'wb') as f:
    pickle.dump(train_sequences, f)
    
with open('../data/test_sequences.pkl', 'wb') as f:
    pickle.dump(test_sequences, f)

# Create ID to Index Mappings

In [13]:
# Get all unique users and movies from the data
all_users = list(user_sequences.keys())
all_movies_list = []
for seq in user_sequences.values():
    # seq is a tuple: (movieID, rating)
    for movie_id, rating in seq: 
        all_movies_list.append(movie_id)
all_movies = list(set(all_movies_list))

print(f"Total unique users: {len(all_users)}")
print(f"Total unique movies: {len(all_movies)}")

Total unique users: 6040
Total unique movies: 3706


In [14]:
# Create mappings
# user2idx has all users from either train or test
user2idx = {user: idx for idx, user in enumerate(all_users)}
idx2user = {idx: user_id for user_id, idx in user2idx.items()}

movie2idx = {movie: idx for idx, movie in enumerate(all_movies)}
idx2movie = {idx: movie for movie, idx in movie2idx.items()}

# Save
pickle.dump(user2idx, open("../data/user2idx.pkl", "wb"))
pickle.dump(idx2user, open("../data/idx2user.pkl", "wb"))
pickle.dump(movie2idx, open("../data/movie2idx.pkl", "wb"))
pickle.dump(idx2movie, open("../data/idx2movie.pkl", "wb"))

print(f"User index range: 0 to {len(user2idx)-1}")
print(f"Movie index range: 0 to {len(movie2idx)-1}")
print(f"ID index range: 0 to {len(idx2movie)-1}")

User index range: 0 to 6039
Movie index range: 0 to 3705
ID index range: 0 to 3705


# More Mappings

In [15]:
def convert_to_idx(sequences_dict, user2idx, movie2idx):
    """
    Convert sequences from original IDs to indices.
    
    Args:
        sequences_dict: dict with user_id keys, values are lists of (movie_id, rating) tuples
        user2idx: mapping from user_id to index
        movie2idx: mapping from movie_id to index
    
    Returns:
        dict with user_idx keys, values are lists of movie indices
    """
    idx_dict = {}
    for user_id, movie_list in sequences_dict.items():
        if user_id in user2idx:
            user_idx = user2idx[user_id]

            movie_indices = []
            for m in movie_list:
                # first element of tuple (movie_id, rating)
                movie_id = m[0]
                if movie_id in movie2idx:
                    movie_idx = movie2idx[movie_id]
                    movie_indices.append(movie_idx)
            
            idx_dict[user_idx] = movie_indices
    return idx_dict

In [16]:
# Convert full user sequences to indices
# Keys: User ID INDICES (NOT user IDs)
# Values: list of Movie Indices (not movie IDs)
user_sequences_idx = convert_to_idx(user_sequences, user2idx, movie2idx)
train_sequences_idx = convert_to_idx(train_sequences, user2idx, movie2idx)
test_sequences_idx = convert_to_idx(test_sequences, user2idx, movie2idx)

In [17]:
pickle.dump(user_sequences_idx, open("../data/user_sequences_idx.pkl", "wb"))
pickle.dump(train_sequences_idx, open("../data/train_sequences_idx.pkl", "wb"))
pickle.dump(test_sequences_idx, open("../data/test_sequences_idx.pkl", "wb"))

# Create data to train the Probabilistic Matrix Factorization model aka the Environment Simulator

In [18]:
print(len(user2idx.keys()))
print(len(movie2idx.keys()))

6040
3706


In [19]:
# Map user and movie IDs to our indices
# First, ensure we only include users and movies that exist in our positive data
valid_users = set(user2idx.keys())
valid_movies = set(movie2idx.keys())

print(len(valid_users))
print(len(valid_movies))

6040
3706


In [20]:
# Filter original ratings to only include valid users and movies
# pmf_data is a DataFrame containing ALL ratings from every user ID in user2idx
pmf_data = ratings[ratings['user_id'].isin(valid_users) & 
                   ratings['movie_id'].isin(valid_movies)].copy()

print(f"Original ratings: {len(ratings)}")
print(f"Ratings for PMF (valid users/movies): {len(pmf_data)}")
print(f"\nRating distribution in PMF data:")
print(pmf_data['rating'].value_counts().sort_index())

Original ratings: 1000209
Ratings for PMF (valid users/movies): 1000209

Rating distribution in PMF data:
1     56174
2    107557
3    261197
4    348971
5    226310
Name: rating, dtype: int64


In [21]:
# Ensure ratings are floats for the model
pmf_data['rating'] = pmf_data['rating'].astype(float)

# Add 2 NEW columns: user_idx and movie_idx so it knows the mapped indices
pmf_data['user_idx'] = pmf_data['user_id'].map(user2idx)
pmf_data['movie_idx'] = pmf_data['movie_id'].map(movie2idx)

# Save
pmf_data.to_csv('../data/movielens_1M_pmf_raw_data.csv', index=False)
print(f"Saved {len(pmf_data)} ratings for PMF pretraining")

Saved 1000209 ratings for PMF pretraining


In [22]:
# Create the lookup dictionary using your actual variable name
ratings_dict = {}
for _, row in pmf_data.iterrows():
    u_idx = user2idx.get(row['user_id'])
    m_idx = movie2idx.get(row['movie_id'])
    if u_idx is not None and m_idx is not None:
        ratings_dict[(u_idx, m_idx)] = row['rating']

print(f"Created lookup dictionary with {len(ratings_dict)} entries.")

Created lookup dictionary with 1000209 entries.


In [23]:
# Save the dictionary to a file
with open('../data/ratings_dict.pkl', 'wb') as f:
    pickle.dump(ratings_dict, f)

In [24]:
# Create a mapping dictionary for quick lookup
movie_id_to_title = dict(zip(movies['movie_id'], movies['title']))

with open('../data/movie_id_to_title.pkl', 'wb') as f:
    pickle.dump(movie_id_to_title, f)

In [25]:
# movie index -> title
movie_idx_to_title = {}
for index, movie_id in idx2movie.items():
    movie_idx_to_title[index] = movie_id_to_title[movie_id]

with open('../data/movie_index_to_title.pkl', 'wb') as f:
    pickle.dump(movie_idx_to_title, f)

# Split 80% Train into 80% train and 20% valid

In [26]:
# Load the existing train_sequences (original IDs)
with open('../data/train_sequences.pkl', 'rb') as f:
    train_sequences = pickle.load(f)

# Optionally load the index version if you want to split that too
with open('../data/train_sequences_idx.pkl', 'rb') as f:
    train_sequences_idx = pickle.load(f)

# Split ratio within the current train set
VAL_RATIO = 0.2   # 20% of current train -> 16% of total

new_train_sequences = {}
valid_sequences = {}
new_train_sequences_idx = {}
valid_sequences_idx = {}

for user, seq in train_sequences.items():
    n = len(seq)
    split_idx = int(n * (1 - VAL_RATIO))   # e.g., 80% of train sequence
    
    # Handle edge cases
    if split_idx == 0:
        split_idx = 1
    if split_idx == n:
        split_idx = n - 1   # leave at least one for validation
    
    new_train_sequences[user] = seq[:split_idx]
    valid_sequences[user] = seq[split_idx:]
    
    # Do the same for the index version if it exists
    if user in train_sequences_idx:
        idx_seq = train_sequences_idx[user]
        new_train_sequences_idx[user] = idx_seq[:split_idx]
        valid_sequences_idx[user] = idx_seq[split_idx:]

# Save the new splits
with open('../data/train_sequences_64.pkl', 'wb') as f:
    pickle.dump(new_train_sequences, f)
with open('../data/valid_sequences.pkl', 'wb') as f:
    pickle.dump(valid_sequences, f)

with open('../data/train_sequences_idx_64.pkl', 'wb') as f:
    pickle.dump(new_train_sequences_idx, f)
with open('../data/valid_sequences_idx.pkl', 'wb') as f:
    pickle.dump(valid_sequences_idx, f)

# Sanity check: total number of ratings in each split
total_train = sum(len(seq) for seq in new_train_sequences.values())
total_valid = sum(len(seq) for seq in valid_sequences.values())
total_test = sum(len(seq) for seq in test_sequences.values())  # test_sequences already exists
print(f"New train ratings: {total_train} ({total_train / (total_train+total_valid+total_test)*100:.1f}%)")
print(f"Validation ratings: {total_valid} ({total_valid / (total_train+total_valid+total_test)*100:.1f}%)")
print(f"Test ratings: {total_test} ({total_test / (total_train+total_valid+total_test)*100:.1f}%)")

New train ratings: 635723 (63.6%)
Validation ratings: 162035 (16.2%)
Test ratings: 202451 (20.2%)


In [27]:
# Things you'll load in other files:
# user_sequences
# train_sequences
# test_sequences
# user2idx
# movie2idx
# idx2movie
# movielens_1M_pmf_data.csv
# ratings_dict
# movie_id_to_title
# user_sequences_idx
# train_sequences_idx
# test_sequences_idx

# Sanity Check

In [28]:
# # Find users with no test data
# users_without_test = []
# for user in train_sequences:
#     if user not in test_sequences or len(test_sequences[user]) == 0:
#         users_without_test.append(user)
        
# print(f"Users without test data: {len(users_without_test)}")
# print(f"Users without test data: {users_without_test}")

In [29]:
# # Examine users
# if len(users_without_test):
#     for user in users_without_test:
#         print(f"\nUser {user}:")
#         print(f"Full sequence length: {len(user_sequences[user])}")
#         print(f"Train sequence: {train_sequences[user]}")
# else:
#     print("Empty!")

In [30]:
# # Check the split logic for these users
# if len(users_without_test):
#     for user in users_without_test:
#         n = len(user_sequences[user])
#         split_point = int(n * TRAIN_RATIO)
#         split_point = max(1, min(split_point, n-1))
        
#         print(f"\nUser {user}: n={n}, split_point={split_point}")
#         print(f"Train (first {split_point}): {user_sequences[user][:split_point]}")
#         print(f"Test (remaining {n-split_point}): {user_sequences[user][split_point:]}")
        
#         # If test is empty, the split_point must equal n
#         if split_point == n:
#             print(f"Issue: split_point = n, so no items left for test")
# else:
#     print("Empty!")